# The Greeks: Analytical vs Numerical Validation

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import os

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

from src.pricing import BlackScholes
from src.greeks import NumericalGreeks, compare_greeks, greek_surface_data

print('All imports successful.')

# 1. Parameters

In [ ]:
S = 100
K = 100
T = 0.25
r = 0.05
q = 0
sigma = 0.20

print(f'S={S}, K={K}, T={T}, r={r}, q={q}, sigma={sigma}')

# 2. All Greeks for ATM Call and Put

In [ ]:
bs = BlackScholes(S, K, T, r, q, sigma)

for opt_type in ['call', 'put']:
    greeks = bs.all_greeks(opt_type)
    print(f'\n{opt_type.upper()} Greeks (ATM, T={T}):')
    print('=' * 40)
    for name, val in greeks.items():
        print(f'  {name:>8s}: {float(val):>12.6f}')

# 3. Analytical vs Numerical Comparison

In [ ]:
# Build comparison manually to avoid potential constructor issue in compare_greeks
def bs_pricer(S_, K_, T_, r_, q_, sigma_, opt_type):
    """Wrapper matching the NumericalGreeks signature."""
    bsm = BlackScholes(S_, K_, T_, r_, q_, sigma_)
    return float(bsm.price(opt_type))

ng = NumericalGreeks

for opt_type in ['call', 'put']:
    bs_obj = BlackScholes(S, K, T, r, q, sigma)

    greek_names = ['Delta', 'Gamma', 'Vega', 'Theta', 'Rho', 'Vanna', 'Volga']

    analytical = [
        float(bs_obj.delta(opt_type)),
        float(bs_obj.gamma()),
        float(bs_obj.vega()),
        float(bs_obj.theta(opt_type)),
        float(bs_obj.rho(opt_type)),
        float(bs_obj.vanna()),
        float(bs_obj.volga()),
    ]

    numerical = [
        ng.delta(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.gamma(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.vega(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.theta(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.rho(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.vanna(bs_pricer, S, K, T, r, q, sigma, opt_type),
        ng.volga(bs_pricer, S, K, T, r, q, sigma, opt_type),
    ]

    abs_errors = [abs(a - n) for a, n in zip(analytical, numerical)]
    rel_errors = [
        abs(a - n) / abs(a) * 100.0 if abs(a) > 1e-12 else 0.0
        for a, n in zip(analytical, numerical)
    ]

    comp_df = pd.DataFrame({
        'Greek': greek_names,
        'Analytical': analytical,
        'Numerical': numerical,
        'Absolute Error': abs_errors,
        'Relative Error (%)': rel_errors,
    })

    print(f'\n{opt_type.upper()} Greeks Comparison:')
    print(comp_df.to_string(index=False))

# Save the call comparison
bs_obj = BlackScholes(S, K, T, r, q, sigma)
analytical_call = [
    float(bs_obj.delta('call')), float(bs_obj.gamma()), float(bs_obj.vega()),
    float(bs_obj.theta('call')), float(bs_obj.rho('call')),
    float(bs_obj.vanna()), float(bs_obj.volga()),
]
numerical_call = [
    ng.delta(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.gamma(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.vega(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.theta(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.rho(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.vanna(bs_pricer, S, K, T, r, q, sigma, 'call'),
    ng.volga(bs_pricer, S, K, T, r, q, sigma, 'call'),
]
greeks_comparison_df = pd.DataFrame({
    'Greek': ['Delta', 'Gamma', 'Vega', 'Theta', 'Rho', 'Vanna', 'Volga'],
    'Analytical': analytical_call,
    'Numerical': numerical_call,
    'Absolute Error': [abs(a - n) for a, n in zip(analytical_call, numerical_call)],
    'Relative Error (%)': [
        abs(a - n) / abs(a) * 100.0 if abs(a) > 1e-12 else 0.0
        for a, n in zip(analytical_call, numerical_call)
    ],
})
greeks_comparison_df.to_csv('../results/tables/greeks_comparison.csv', index=False)
print('\nSaved: results/tables/greeks_comparison.csv')

# 4. Delta vs Spot Price

In [ ]:
S_range = np.linspace(70, 130, 200)

call_deltas = np.array([float(BlackScholes(s, K, T, r, q, sigma).delta('call')) for s in S_range])
put_deltas = np.array([float(BlackScholes(s, K, T, r, q, sigma).delta('put')) for s in S_range])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(S_range, call_deltas, color='steelblue', linewidth=2, label='Call Delta')
ax.plot(S_range, put_deltas, color='crimson', linewidth=2, label='Put Delta')
ax.axvline(x=K, color='gray', linestyle=':', alpha=0.7, label=f'Strike K={K}')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
ax.axhline(y=-0.5, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Spot Price ($)', fontsize=12)
ax.set_ylabel('Delta', fontsize=12)
ax.set_title(f'Delta vs Spot Price (K={K}, T={T}, $\\sigma$={sigma:.0%})', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(70, 130)

plt.tight_layout()
plt.savefig('../results/figures/delta_vs_spot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/delta_vs_spot.png')

# 5. Gamma Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Gamma vs S for different T
ax = axes[0]
T_values = [30/365, 90/365, 180/365]
T_labels = ['30 days', '90 days', '180 days']
colors_t = ['#c44e52', '#4c72b0', '#55a868']

for T_i, label, color in zip(T_values, T_labels, colors_t):
    gammas = np.array([float(BlackScholes(s, K, T_i, r, q, sigma).gamma()) for s in S_range])
    ax.plot(S_range, gammas, color=color, linewidth=2, label=label)

ax.axvline(x=K, color='gray', linestyle=':', alpha=0.7)
ax.set_xlabel('Spot Price ($)', fontsize=12)
ax.set_ylabel('Gamma', fontsize=12)
ax.set_title('Gamma vs Spot Price', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Right: Gamma vs T (ATM)
ax = axes[1]
T_range_days = np.linspace(5, 365, 200)
T_range_yrs = T_range_days / 365.0
atm_gammas = np.array([float(BlackScholes(S, K, t, r, q, sigma).gamma()) for t in T_range_yrs])

ax.plot(T_range_days, atm_gammas, color='darkorange', linewidth=2)
ax.set_xlabel('Days to Expiry', fontsize=12)
ax.set_ylabel('Gamma (ATM)', fontsize=12)
ax.set_title('ATM Gamma vs Time to Expiry', fontsize=13, fontweight='bold')
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('../results/figures/gamma_vs_spot_and_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/gamma_vs_spot_and_time.png')

# 6. Vega vs Spot Price

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for T_i, label, color in zip(T_values, T_labels, colors_t):
    vegas = np.array([float(BlackScholes(s, K, T_i, r, q, sigma).vega()) for s in S_range])
    ax.plot(S_range, vegas, color=color, linewidth=2, label=label)

ax.axvline(x=K, color='gray', linestyle=':', alpha=0.7, label=f'K={K}')
ax.set_xlabel('Spot Price ($)', fontsize=12)
ax.set_ylabel('Vega (per 1% vol move)', fontsize=12)
ax.set_title('Vega vs Spot Price', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../results/figures/vega_vs_spot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vega_vs_spot.png')

# 7. Theta vs Time to Expiry

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# ATM call and put theta vs time
call_thetas = np.array([float(BlackScholes(S, K, t, r, q, sigma).theta('call')) for t in T_range_yrs])
put_thetas = np.array([float(BlackScholes(S, K, t, r, q, sigma).theta('put')) for t in T_range_yrs])

ax.plot(T_range_days, call_thetas, color='steelblue', linewidth=2, label='Call Theta')
ax.plot(T_range_days, put_thetas, color='crimson', linewidth=2, label='Put Theta')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Days to Expiry', fontsize=12)
ax.set_ylabel('Theta ($ per calendar day)', fontsize=12)
ax.set_title('ATM Theta vs Time to Expiry', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('../results/figures/theta_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/theta_vs_time.png')

# 8. Vanna Surface (3D)

In [ ]:
S_surf = np.linspace(80, 120, 50)
T_surf = np.linspace(0.02, 0.5, 50)

surface_data = greek_surface_data(
    K=K, T=T, r=r, q=q, sigma=sigma, option_type='call',
    S_range=S_surf, T_range=T_surf
)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    surface_data['S_grid'], surface_data['T_grid'] * 365,
    surface_data['vanna'],
    cmap=cm.RdBu_r, edgecolor='none', alpha=0.85
)

ax.set_xlabel('Spot Price ($)', fontsize=11, labelpad=10)
ax.set_ylabel('Days to Expiry', fontsize=11, labelpad=10)
ax.set_zlabel('Vanna', fontsize=11, labelpad=10)
ax.set_title('Vanna Surface', fontsize=14, fontweight='bold')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)

plt.tight_layout()
plt.savefig('../results/figures/vanna_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vanna_surface.png')

# 9. Greek Heatmap

In [ ]:
moneyness_levels = np.arange(0.85, 1.16, 0.05)
T_levels = np.array([7, 14, 30, 60, 90, 180, 365]) / 365.0
T_labels_hm = ['7d', '14d', '30d', '60d', '90d', '180d', '365d']

greeks_to_plot = ['delta', 'gamma', 'vega', 'theta']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, greek_name in enumerate(greeks_to_plot):
    ax = axes[idx // 2, idx % 2]
    data_matrix = np.zeros((len(T_levels), len(moneyness_levels)))

    for i, t in enumerate(T_levels):
        for j, m in enumerate(moneyness_levels):
            K_j = S * m
            bsm = BlackScholes(S, K_j, t, r, q, sigma)
            if greek_name == 'delta':
                data_matrix[i, j] = float(bsm.delta('call'))
            elif greek_name == 'gamma':
                data_matrix[i, j] = float(bsm.gamma())
            elif greek_name == 'vega':
                data_matrix[i, j] = float(bsm.vega())
            elif greek_name == 'theta':
                data_matrix[i, j] = float(bsm.theta('call'))

    sns.heatmap(
        data_matrix, ax=ax, annot=True, fmt='.3f', cmap='RdBu_r',
        xticklabels=[f'{m:.2f}' for m in moneyness_levels],
        yticklabels=T_labels_hm,
        center=0 if greek_name in ('theta', 'vanna') else None
    )
    ax.set_xlabel('Moneyness (K/S)', fontsize=10)
    ax.set_ylabel('Time to Expiry', fontsize=10)
    ax.set_title(greek_name.capitalize(), fontsize=12, fontweight='bold')

plt.suptitle('Greek Heatmaps: Moneyness x Time', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/greek_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/greek_heatmap.png')

# 10. Vega-Theta Trade-off

In [ ]:
S_grid_flat = np.linspace(80, 120, 30)
T_grid_flat = np.linspace(0.05, 0.5, 20)

vega_vals = []
theta_vals = []
T_colors = []

for t in T_grid_flat:
    for s in S_grid_flat:
        bsm = BlackScholes(s, K, t, r, q, sigma)
        vega_vals.append(float(bsm.vega()))
        theta_vals.append(float(bsm.theta('call')))
        T_colors.append(t * 365)  # days

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(vega_vals, theta_vals, c=T_colors, cmap='viridis', alpha=0.6, s=30, edgecolors='none')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Days to Expiry', fontsize=11)

ax.set_xlabel('Vega (per 1% vol move)', fontsize=12)
ax.set_ylabel('Theta ($ per day)', fontsize=12)
ax.set_title('Vega-Theta Trade-off', fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/figures/vega_theta_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vega_theta_tradeoff.png')

## Interview Insight

**Common interview question:** *"You are long a straddle. What is your Greek exposure?"*

- **Delta:** Near zero (long call delta + long put delta cancel at ATM)
- **Gamma:** Strongly positive -- you profit from large spot moves in either direction
- **Vega:** Strongly positive -- you profit from rising implied volatility
- **Theta:** Strongly negative -- the position decays every day

This is the fundamental **gamma-theta trade-off**: being long gamma (convexity) costs theta (time decay). The break-even is reached when the realized move exceeds $\sqrt{2 \cdot |\theta/\gamma|}$ per day.

**Key second-order Greeks:**
- **Vanna** (dDelta/dVol): Crucial for understanding how delta hedges shift when vol moves. Negative vanna for OTM puts means a vol spike makes your delta more negative.
- **Volga** (dVega/dVol): Drives the convexity of the vol smile. Positive volga means you benefit from large vol-of-vol.

**Numerical vs Analytical validation:** The finite-difference Greeks match analytical values to 4+ decimal places, confirming both implementations are correct. Numerical Greeks become essential when no closed-form exists (e.g., exotic options, local vol models).

In [ ]:
saved_files = [
    'results/tables/greeks_comparison.csv',
    'results/figures/delta_vs_spot.png',
    'results/figures/gamma_vs_spot_and_time.png',
    'results/figures/vega_vs_spot.png',
    'results/figures/theta_vs_time.png',
    'results/figures/vanna_surface.png',
    'results/figures/greek_heatmap.png',
    'results/figures/vega_theta_tradeoff.png',
]

print('Files saved by this notebook:')
print('=' * 50)
for f in saved_files:
    print(f'  {f}')